### Overview

Cleans the TCGA-BRCA dataset by removing duplicated patient records and normal-like subtype.

In [1]:
import pandas as pd
import numpy as np
import random

### 1. Import data

In [2]:
raw_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_uncleaned_data/geneExp_unstranded.csv" header = 0,
                         index_col = 0).transpose()

In [3]:
tpm_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_uncleaned_data/geneExp_tpm_unstrand.csv", header = 0,
                         index_col = 0).transpose()

In [4]:
sample_info = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_uncleaned_data/sample_info.csv", header = 0,
                         index_col = 0)

In [5]:
sample_info.shape

(1111, 85)

### 2. Data cleaning

In [6]:
# check if the indexes of all the files match
print(raw_counts.index.equals(sample_info.index))
print(tpm_counts.index.equals(sample_info.index))
print(tpm_counts.index.equals(raw_counts.index))

True
True
True


In [7]:
# find the number of unique patient IDs
sample_info['patient'].nunique()

1095

In [8]:
# get the subset of patients and their duplicates
patient_duplicates_with_first_occ = sample_info.loc[sample_info['patient'].duplicated(keep=False)]

In [9]:
# count of all duplicates including first occurrence
patient_duplicates_with_first_occ.shape[0]

27

In [10]:
# # check if the counts of the duplicates are the same in raw, tpm and fpkm reads
# raw_counts.loc[patient_duplicates_with_first_occ.index].sort_index().head(3)
# tpm_counts.loc[patient_duplicates_with_first_occ.index].sort_index().head(3)

In [11]:
# remove all duplicates including first occurrences to avoid bias because counts are different across the duplicates
sample_info_cleaned = sample_info.drop_duplicates(subset = ['patient'], keep = False)
raw_counts_cleaned = raw_counts.loc[sample_info_cleaned.index]
tpm_counts_cleaned = tpm_counts.loc[sample_info_cleaned.index]

In [12]:
# check the count of each gender in the sample info without duplicates
sample_info_cleaned['gender'].value_counts()

gender
female    1071
male        12
Name: count, dtype: int64

In [13]:
# keep only female samples in the dataframes without duplicates
sample_info_cleaned = sample_info_cleaned.loc[sample_info_cleaned['gender'] == 'female']
raw_counts_cleaned = raw_counts_cleaned.loc[sample_info_cleaned.index]
tpm_counts_cleaned = tpm_counts_cleaned.loc[sample_info_cleaned.index]

In [14]:
# check the count of samples with missing subtype information
sample_info_cleaned['paper_BRCA_Subtype_PAM50'].isna().sum()

0

In [15]:
# check the count of each subtype
sample_info_cleaned['paper_BRCA_Subtype_PAM50'].value_counts()

paper_BRCA_Subtype_PAM50
LumA      555
LumB      209
Basal     185
Her2       82
Normal     40
Name: count, dtype: int64

In [16]:
# remove normal-like subtype
sample_info_cleaned = sample_info_cleaned.loc[sample_info_cleaned['paper_BRCA_Subtype_PAM50'] != 'Normal']
raw_counts_cleaned = raw_counts_cleaned.loc[sample_info_cleaned.index]
tpm_counts_cleaned = tpm_counts_cleaned.loc[sample_info_cleaned.index]

In [17]:
# check the dimensions of the new dataframes without duplicates
print(sample_info_cleaned.shape)
print(raw_counts_cleaned.shape)
print(tpm_counts_cleaned.shape)

(1031, 85)
(1031, 60660)
(1031, 60660)
(1031, 60660)


In [16]:
# check if the tpm counts are really tpm by checking the sum of transcripts for each sample
tpm_counts_cleaned.sum(axis=1)

TCGA-D8-A146-01A-31R-A115-07    1.000000e+06
TCGA-AQ-A0Y5-01A-11R-A14M-07    1.000000e+06
TCGA-C8-A274-01A-11R-A16F-07    1.000000e+06
TCGA-BH-A0BD-01A-11R-A034-07    1.000000e+06
TCGA-B6-A1KC-01B-11R-A157-07    1.000000e+06
                                    ...     
TCGA-BH-A0EI-01A-11R-A115-07    1.000000e+06
TCGA-E2-A1IO-01A-11R-A144-07    1.000000e+06
TCGA-E2-A15R-01A-11R-A115-07    1.000000e+06
TCGA-B6-A0IP-01A-11R-A034-07    1.000000e+06
TCGA-A1-A0SN-01A-11R-A144-07    1.000000e+06
Length: 1031, dtype: float64

In [18]:
# check if the indexes of the new dataframes match
print(raw_counts_cleaned.index.equals(sample_info_cleaned.index))
print(tpm_counts_cleaned.index.equals(sample_info_cleaned.index))
print(tpm_counts_cleaned.index.equals(raw_counts_cleaned.index))

True
True
True


In [9]:
# # save cleaned data as csv
# sample_info_cleaned.to_csv('cleaned sample info.csv')
# raw_counts_cleaned.to_csv('cleaned raw counts.csv')
# tpm_counts_cleaned.to_csv('cleaned tpm counts.csv')

### 3. Checking other clinical data

In [21]:
sample_info_cleaned['vital_status'].value_counts()

vital_status
Alive    888
Dead     143
Name: count, dtype: int64

In [22]:
sample_info_cleaned['paper_pathologic_stage'].value_counts()

paper_pathologic_stage
Stage_II     582
Stage_III    234
Stage_I      173
Stage_IV      18
Name: count, dtype: int64

In [23]:
sample_info_cleaned['paper_pathologic_stage'].isna().sum()

24

In [24]:
sample_info_cleaned['paper_BRCA_Pathology'].value_counts()

paper_BRCA_Pathology
IDC      500
ILC      120
Other    110
Mixed     91
Name: count, dtype: int64

In [25]:
sample_info_cleaned['paper_BRCA_Pathology'].isna().sum()

210

In [26]:
sample_info_cleaned['preservation_method'].value_counts()

preservation_method
OCT        640
Unknown    387
FFPE         4
Name: count, dtype: int64

In [42]:
sample_info_cleaned['is_ffpe'].value_counts()

is_ffpe
False    1030
True        1
Name: count, dtype: int64

In [43]:
sample_info_cleaned['oct_embedded'].value_counts()

oct_embedded
True     640
False    391
Name: count, dtype: int64

In [41]:
# check which columns in sample info are not strings (objects) and if they have missing values
for i in range(len(sample_info_cleaned.columns)):
    if  sample_info_cleaned.dtypes.iloc[i] != "object":
        print(f"column {sample_info_cleaned.columns[i]} is {sample_info_cleaned[sample_info_cleaned.columns[i]].dtype}.")
        print(f"missing values in this column: {sample_info_cleaned[sample_info_cleaned.columns[i]].isna().sum()}\n")

column sample_type_id is int64.
missing values in this column: 0

column days_to_collection is float64.
missing values in this column: 2

column initial_weight is float64.
missing values in this column: 2

column oct_embedded is bool.
missing values in this column: 0

column is_ffpe is bool.
missing values in this column: 0

column days_to_diagnosis is float64.
missing values in this column: 0

column days_to_last_follow_up is float64.
missing values in this column: 99

column age_at_diagnosis is float64.
missing values in this column: 15

column year_of_diagnosis is float64.
missing values in this column: 2

column age_at_index is float64.
missing values in this column: 0

column days_to_birth is float64.
missing values in this column: 15

column year_of_birth is float64.
missing values in this column: 3

column days_to_death is float64.
missing values in this column: 889

column year_of_death is float64.
missing values in this column: 932

column releasable is bool.
missing values in